Install and Load SQL utilities

In [ ]:
%pip install jupysql sqlalchemy

In [1]:
%load_ext sql

# 30 Day Readmission Risk Analysis
### Notebook 1: SQL Prep and Cleaning

Connect and initialize SQLITE database, then set number of display rows for output to 50.

In [2]:
%sql sqlite:///readmissions/readmissions.db

Connecting to 'sqlite:///readmissions/readmissions.db'

In [3]:
%config SqlMagic.displaylimit = 50

#### Import CSVs to SQLITE database
We have two tables, the main readmissions data table, and the ICD 10 chapter code lookup table. These are in CSV format and must be imported at the command line.

In [4]:
!sqlite3 readmissions/readmissions.db ".mode csv" ".import --skip 0 readmissions/ICD10ChapterCodes.csv icd_10_chapter_codes"

In [5]:
!sqlite3 readmissions/readmissions.db ".mode csv" ".import --skip 0 readmissions/HealthGuard_Readmission_Data/readmission_dataset.csv readmissions_raw"

Confirm that the raw tables exist and the columns and data look usable

In [6]:
%%sql
SELECT
  name
FROM
  sqlite_schema
WHERE
  type = 'table'
  AND name NOT LIKE 'sqlite_%';

Running query in 'sqlite:///readmissions/readmissions.db'

name
icd_10_chapter_codes
readmissions_raw


In [10]:
%%sql
SELECT
  *
FROM
  readmissions_raw
LIMIT
  10;

Running query in 'sqlite:///readmissions/readmissions.db'

patient_id,age,gender,admission_type,primary_diagnosis_code,num_prior_admissions,time_in_hospital,num_lab_procedures,num_medications,has_comorbidity,discharge_disposition,insurance_type,hospital_id,readmitted_within_30days,days_to_readmission
1,75,Other,Emergency,E11,0,5,88.0,35.0,1,Home,Medicare,5,1,7.0
2,72,Female,Elective,J45,0,6,46.0,24.0,0,Rehabilitation,Private,5,0,
3,73,Female,Urgent,J45,0,3,22.0,3.0,0,Transfer,Private,3,0,
4,61,Male,Elective,G47,0,4,14.0,4.0,1,Rehabilitation,Private,6,0,
5,87,Female,Urgent,N39,3,6,35.0,28.0,1,Home,Private,3,1,9.0
6,59,Female,Emergency,E11,0,1,5.0,8.0,1,Rehabilitation,Private,7,0,
7,80,Male,Urgent,I10,0,2,86.0,24.0,0,Rehabilitation,Private,1,0,
8,66,Female,Urgent,K21,0,3,89.0,10.0,0,Home,Private,4,0,
9,70,Male,Elective,F32,2,7,98.0,22.0,0,Home,Private,2,0,
10,81,Male,Emergency,G47,0,8,34.0,7.0,1,Home,Medicaid,5,0,


Check column names and data types in case we have to recast, and we do.

In [11]:
%%sql
PRAGMA table_info (readmissions_raw)

Running query in 'sqlite:///readmissions/readmissions.db'

cid,name,type,notnull,dflt_value,pk
0,patient_id,TEXT,0,None,0
1,age,TEXT,0,None,0
2,gender,TEXT,0,None,0
3,admission_type,TEXT,0,None,0
4,primary_diagnosis_code,TEXT,0,None,0
5,num_prior_admissions,TEXT,0,None,0
6,time_in_hospital,TEXT,0,None,0
7,num_lab_procedures,TEXT,0,None,0
8,num_medications,TEXT,0,None,0
9,has_comorbidity,TEXT,0,None,0


In [13]:
%%sql
PRAGMA table_info (icd_10_chapter_codes)

Running query in 'sqlite:///readmissions/readmissions.db'

cid,name,type,notnull,dflt_value,pk
0,chapter_code,TEXT,0,None,0
1,diagnosis_desc,TEXT,0,None,0


Check row counts for completeness of data

In [15]:
%%sql
SELECT
  COUNT(patient_id)
FROM
  readmissions_raw;

Running query in 'sqlite:///readmissions/readmissions.db'

COUNT(patient_id)
10000


Create new table for cleaned data

In [8]:
%%sql
CREATE TABLE
  IF NOT EXISTS readmissions_typed (
    patient_id INTEGER PRIMARY KEY,
    age INTEGER,
    gender TEXT,
    admission_type TEXT,
    primary_diagnosis_code TEXT,
    num_prior_admissions INTEGER,
    time_in_hospital INTEGER,
    num_lab_procedures INTEGER,
    num_medications INTEGER,
    has_comorbidity INTEGER,
    discharge_disposition TEXT,
    insurance_type TEXT,
    hospital_id INTEGER,
    readmitted_within_30days INTEGER,
    days_to_readmission REAL
  );

Running query in 'sqlite:///readmissions/readmissions.db'

++
||
++
++

In [4]:
%%sql
SELECT
  name
FROM
  sqlite_schema
WHERE
  type = 'table'
  AND name NOT LIKE 'sqlite_%';

Running query in 'sqlite:///readmissions/readmissions.db'

name
icd_10_chapter_codes
readmissions_raw
readmissions_typed


#### Cast data as correct data types to facilitate cleaning and analysis

In [10]:
%%sql
INSERT INTO
  readmissions_typed (
    patient_id,
    age,
    gender,
    admission_type,
    primary_diagnosis_code,
    num_prior_admissions,
    time_in_hospital,
    num_lab_procedures,
    num_medications,
    has_comorbidity,
    discharge_disposition,
    insurance_type,
    hospital_id,
    readmitted_within_30days,
    days_to_readmission
  )
SELECT
  CAST(patient_id AS INTEGER),
  CAST(age AS INTEGER),
  gender,
  admission_type,
  primary_diagnosis_code,
  CAST(num_prior_admissions AS INTEGER),
  CAST(time_in_hospital AS INTEGER),
  CAST(num_lab_procedures AS INTEGER),
  CAST(num_medications AS INTEGER),
  CAST(has_comorbidity AS INTEGER),
  discharge_disposition,
  insurance_type,
  CAST(hospital_id AS INTEGER),
  CAST(readmitted_within_30days AS INTEGER),
  CAST(days_to_readmission AS REAL)
FROM
  readmissions_raw;

Running query in 'sqlite:///readmissions/readmissions.db'

10000 rows affected.

++
||
++
++

#### Data Cleaning
Ensure that any missing data is cast as null.

In [11]:
%%sql
UPDATE readmissions_typed
SET
  age = CASE
    WHEN age IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN age < 0 THEN NULL
    ELSE age
  END,
  gender = CASE
    WHEN TRIM(LOWER(gender)) IN ('', 'unknown', 'nan', 'error') THEN NULL
    ELSE TRIM(LOWER(gender))
  END,
  admission_type = CASE
    WHEN TRIM(LOWER(admission_type)) IN ('', 'unknown', 'nan', 'error') THEN NULL
    ELSE TRIM(LOWER(admission_type))
  END,
  primary_diagnosis_code = CASE
    WHEN TRIM(LOWER(primary_diagnosis_code)) IN ('', 'unknown', 'nan', 'error') THEN NULL
    ELSE TRIM(LOWER(primary_diagnosis_code))
  END,
  num_prior_admissions = CASE
    WHEN num_prior_admissions IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN num_prior_admissions < 0 THEN NULL
    ELSE num_prior_admissions
  END,
  time_in_hospital = CASE
    WHEN time_in_hospital IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN time_in_hospital < 0 THEN NULL
    ELSE time_in_hospital
  END,
  num_lab_procedures = CASE
    WHEN num_lab_procedures IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN num_lab_procedures < 0 THEN NULL
    ELSE num_lab_procedures
  END,
  num_medications = CASE
    WHEN num_medications IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN num_medications < 0 THEN NULL
    ELSE num_medications
  END,
  has_comorbidity = CASE
    WHEN has_comorbidity IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN has_comorbidity < 0 THEN NULL
    ELSE has_comorbidity
  END,
  discharge_disposition = CASE
    WHEN TRIM(LOWER(discharge_disposition)) IN ('', 'unknown', 'nan', 'error') THEN NULL
    ELSE TRIM(LOWER(discharge_disposition))
  END,
  insurance_type = CASE
    WHEN TRIM(LOWER(insurance_type)) IN ('', 'unknown', 'nan', 'error') THEN NULL
    ELSE TRIM(LOWER(insurance_type))
  END,
  hospital_id = CASE
    WHEN hospital_id IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN hospital_id < 0 THEN NULL
    ELSE hospital_id
  END,
  readmitted_within_30days = CASE
    WHEN readmitted_within_30days IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN readmitted_within_30days < 0 THEN NULL
    ELSE readmitted_within_30days
  END,
  days_to_readmission = CASE
    WHEN days_to_readmission IN ('', 'unknown', 'nan', 'error') THEN NULL
    WHEN days_to_readmission < 0 THEN NULL
    ELSE days_to_readmission
  END;

Running query in 'sqlite:///readmissions/readmissions.db'

10000 rows affected.

++
||
++
++

Ensure that there are no duplicates on the patient ID field.

In [12]:
%%sql
SELECT
  patient_id,
  COUNT(*) AS COUNT
FROM
  readmissions_raw
GROUP BY
  patient_id
HAVING
  COUNT(*) > 1;

Running query in 'sqlite:///readmissions/readmissions.db'

patient_id,count


Drop any rows that have nulls in the index or the target fields.

In [13]:
%%sql
DELETE FROM readmissions_typed
WHERE
  patient_id IS NULL
  OR readmitted_within_30days IS NULL;

Running query in 'sqlite:///readmissions/readmissions.db'

++
||
++
++

In [15]:
%%sql
SELECT
  *
FROM
  readmissions_typed
LIMIT
  10;

Running query in 'sqlite:///readmissions/readmissions.db'

patient_id,age,gender,admission_type,primary_diagnosis_code,num_prior_admissions,time_in_hospital,num_lab_procedures,num_medications,has_comorbidity,discharge_disposition,insurance_type,hospital_id,readmitted_within_30days,days_to_readmission
1,75,other,emergency,e11,0,5,88,35,1,home,medicare,5,1,7.0
2,72,female,elective,j45,0,6,46,24,0,rehabilitation,private,5,0,0.0
3,73,female,urgent,j45,0,3,22,3,0,transfer,private,3,0,0.0
4,61,male,elective,g47,0,4,14,4,1,rehabilitation,private,6,0,0.0
5,87,female,urgent,n39,3,6,35,28,1,home,private,3,1,9.0
6,59,female,emergency,e11,0,1,5,8,1,rehabilitation,private,7,0,0.0
7,80,male,urgent,i10,0,2,86,24,0,rehabilitation,private,1,0,0.0
8,66,female,urgent,k21,0,3,89,10,0,home,private,4,0,0.0
9,70,male,elective,f32,2,7,98,22,0,home,private,2,0,0.0
10,81,male,emergency,g47,0,8,34,7,1,home,medicaid,5,0,0.0


In [20]:
%%sql
SELECT
  COUNT(patient_id)
FROM
  readmissions_typed;

Running query in 'sqlite:///readmissions/readmissions.db'

COUNT(patient_id)
10000


In [21]:
%%sql
SELECT
  *
FROM
  readmissions_typed
LIMIT
  5;

Running query in 'sqlite:///readmissions/readmissions.db'

patient_id,age,gender,admission_type,primary_diagnosis_code,num_prior_admissions,time_in_hospital,num_lab_procedures,num_medications,has_comorbidity,discharge_disposition,insurance_type,hospital_id,readmitted_within_30days,days_to_readmission
1,75,other,emergency,e11,0,5,88,35,1,home,medicare,5,1,7.0
2,72,female,elective,j45,0,6,46,24,0,rehabilitation,private,5,0,0.0
3,73,female,urgent,j45,0,3,22,3,0,transfer,private,3,0,0.0
4,61,male,elective,g47,0,4,14,4,1,rehabilitation,private,6,0,0.0
5,87,female,urgent,n39,3,6,35,28,1,home,private,3,1,9.0


In [36]:
%%sql
UPDATE icd_10_chapter_codes
SET
  chapter_code = TRIM(LOWER(chapter_code));

Running query in 'sqlite:///readmissions/readmissions.db'

10 rows affected.

++
||
++
++

#### Preliminary Exploratory Data Analysis

What diagnoses do we have in the dataset, and how many patients fall in each?

In [37]:
%%sql
SELECT
  r.primary_diagnosis_code,
  i.diagnosis_desc,
  COUNT(r.patient_id) AS patient_count
FROM
  readmissions_typed r
  INNER JOIN icd_10_chapter_codes i ON r.primary_diagnosis_code = i.chapter_code
GROUP BY
  r.primary_diagnosis_code,
  i.diagnosis_desc
ORDER BY
  patient_count DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

primary_diagnosis_code,diagnosis_desc,patient_count
e11,Type 2 diabetes mellitus,1212
i10,Essential (primary) hypertension,1157
e78,Disorders of lipoprotein metabolism and other lipidemias,1046
f32,Depressive episode,1033
k21,Gastro-esophageal reflux disease,1029
j45,Asthma,996
m54,Dorsalgia,990
n39,Other disorders of urinary system,971
g47,Sleep disorders,821
i25,Chronic ischemic heart disease,745


What is the baseline readmission rate?

In [7]:
%%sql
SELECT
  readmitted_within_30days,
  COUNT(*) AS readmitted_num,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed
GROUP BY
  readmitted_within_30days
ORDER BY
  readmitted_num DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

readmitted_within_30days,readmitted_num,percentage
0,7293,72.0
1,2707,27.0


In [7]:
%%sql
SELECT
  readmitted_within_30days,
  COUNT(*) AS readmitted_num,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed
GROUP BY
  readmitted_within_30days
ORDER BY
  readmitted_num DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

readmitted_within_30days,readmitted_num,percentage
0,7293,72.0
1,2707,27.0


What is the breakdown by gender in the dataset?

In [9]:
%%sql
SELECT
  gender,
  COUNT(*) AS gender_count,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed
GROUP BY
  gender
ORDER BY
  gender_count DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

gender,gender_count,percentage
female,4935,49.0
male,4866,48.0
other,199,1.0


What does our age distribution look like?

In [38]:
%%sql
SELECT 
    CASE
        WHEN age <= 19 THEN '0-19'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        WHEN age BETWEEN 50 AND 59 THEN '50-59'
        WHEN age BETWEEN 60 AND 69 THEN '60-69'
        WHEN age BETWEEN 70 AND 79 THEN '70-79'
        WHEN age BETWEEN 80 AND 89 THEN '80-89'
        ELSE '90+'
    END AS age_group,
    COUNT(*) AS total_count
FROM readmissions_typed
GROUP BY age_group
ORDER by age;

Running query in 'sqlite:///readmissions/readmissions.db'

age_group,total_count
20-29,4
30-39,75
40-49,484
50-59,1382
60-69,2580
70-79,3345
80-89,2121
90+,9


How about admissions type?

In [13]:
%%sql
SELECT
  admission_type,
  COUNT(*) AS adm_type_count,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed
GROUP BY
  admission_type
ORDER BY
  adm_type_count DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

admission_type,adm_type_count,percentage
emergency,5006,50.0
elective,3037,30.0
urgent,1957,19.0


What's the distribution of the number of prior admissions?
For number of prior admissions, roughly half have no prior admissions.

In [10]:
%%sql
SELECT
  num_prior_admissions,
  COUNT(*) AS num_adm_count
FROM
  readmissions_typed
GROUP BY
  num_prior_admissions
ORDER BY
  num_adm_count DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

num_prior_admissions,num_adm_count
0,4906
1,2047
2,1521
3,1046
4,389
5,91


For length of stay, let's first look at a distribution

In [7]:
%%sql
SELECT
  time_in_hospital,
  COUNT(*) AS time_hosp_count
FROM
  readmissions_typed
GROUP BY
  time_in_hospital
ORDER BY
  time_in_hospital DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

time_in_hospital,time_hosp_count
20,1
14,3
13,18
12,29
11,71
10,187
9,350
8,647
7,1016
6,1508


Then let's look at an average length of stay by diagnosis code.

In [15]:
%%sql
SELECT
  r.primary_diagnosis_code,
  i.diagnosis_desc,
  AVG(r.time_in_hospital) AS average_los,
  COUNT(r.patient_id) AS patient_count,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed r
  JOIN icd_10_chapter_codes i ON r.primary_diagnosis_code = i.chapter_code
GROUP BY
  primary_diagnosis_code

Running query in 'sqlite:///readmissions/readmissions.db'

primary_diagnosis_code,diagnosis_desc,average_los,patient_count,percentage
e11,Type 2 diabetes mellitus,5.102310231023102,1212,12.0
e78,Disorders of lipoprotein metabolism and other lipidemias,5.0573613766730405,1046,10.0
f32,Depressive episode,4.9728944820909975,1033,10.0
g47,Sleep disorders,4.97442143727162,821,8.0
i10,Essential (primary) hypertension,5.008643042350908,1157,11.0
i25,Chronic ischemic heart disease,4.994630872483222,745,7.0
j45,Asthma,4.974899598393574,996,9.0
k21,Gastro-esophageal reflux disease,4.997084548104956,1029,10.0
m54,Dorsalgia,4.925252525252525,990,9.0
n39,Other disorders of urinary system,4.797116374871266,971,9.0


And a weighted average length of stay for all diagnoses

In [48]:
%%sql
WITH
  diag_segments AS (
    SELECT
      primary_diagnosis_code,
      AVG(time_in_hospital) AS average_los,
      COUNT(patient_id) AS patient_count
    FROM
      readmissions_typed
    GROUP BY
      primary_diagnosis_code
  )
SELECT
  SUM(average_los * patient_count) / SUM(patient_count) AS weighted_alos
FROM
  diag_segments;

Running query in 'sqlite:///readmissions/readmissions.db'

weighted_alos
4.9842


What does the distribution of days to readmission look like?

In [12]:
%%sql
SELECT 
    CASE
        WHEN days_to_readmission BETWEEN 1 AND 5 THEN '1-5 Days'
        WHEN days_to_readmission BETWEEN 6 AND 10 THEN '6-10 Days'
        WHEN days_to_readmission BETWEEN 11 AND 15 THEN '11-15 Days'
        WHEN days_to_readmission BETWEEN 16 AND 20 THEN '16-20 Days'
        WHEN days_to_readmission BETWEEN 21 AND 25 THEN '21-25 Days'
        WHEN days_to_readmission BETWEEN 26 AND 30 THEN '26-30 Days'
    END AS banded_days_to_readmission,
    COUNT(*) AS total_count
FROM readmissions_typed
GROUP BY banded_days_to_readmission;

Running query in 'sqlite:///readmissions/readmissions.db'

banded_days_to_readmission,total_count
None,7293
1-5 Days,454
11-15 Days,408
16-20 Days,469
21-25 Days,457
26-30 Days,420
6-10 Days,499


How about insurance type?

In [16]:
%%sql
SELECT
  insurance_type,
  COUNT(*) AS ins_type_count,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed
GROUP BY
  insurance_type
ORDER BY
  ins_type_count ASC;

Running query in 'sqlite:///readmissions/readmissions.db'

insurance_type,ins_type_count,percentage
self-pay,460,4.0
medicaid,1468,14.0
medicare,3014,30.0
private,5058,50.0


Are the segments selected from each hospital relatively evenly distributed?

In [17]:
%%sql
SELECT
  hospital_id,
  COUNT(*) AS hosp_id_count,
  ROUND(COUNT(*) * 100 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM
  readmissions_typed
GROUP BY
  hospital_id
ORDER BY
  hosp_id_count DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

hospital_id,hosp_id_count,percentage
3,1478,14.0
1,1455,14.0
5,1445,14.0
2,1433,14.0
4,1407,14.0
6,1401,14.0
7,1381,13.0


Let's look at some descriptive averages for some of the key risk factors for the dataset as a whole.

In [5]:
%%sql
SELECT
  ROUND(AVG(time_in_hospital), 2) AS avg_time_in_hospital,
  ROUND(AVG(num_medications), 2) AS avg_num_medications,
  ROUND(AVG(num_lab_procedures), 2) AS avg_num_lab_procedures,
  ROUND(AVG(num_prior_admissions), 2) AS avg_num_prior_admissions
FROM
  readmissions_typed;

Running query in 'sqlite:///readmissions/readmissions.db'

avg_time_in_hospital,avg_num_medications,avg_num_lab_procedures,avg_num_prior_admissions
4.98,25.09,49.74,1.02


In [7]:
%%sql
SELECT
  admission_type,
  COUNT(DISTINCT patient_id) AS patients,
  SUM(readmitted_within_30days) AS readmitted_30days,
  ROUND(
    100 * SUM(readmitted_within_30days) / COUNT(patient_id),
    2
  ) AS readmission_30_day_rate,
  ROUND(AVG(time_in_hospital), 2) AS avg_time_in_hospital,
  ROUND(AVG(num_medications), 2) AS avg_num_medications,
  ROUND(AVG(num_lab_procedures), 2) AS avg_num_lab_procedures,
  ROUND(AVG(num_prior_admissions), 2) AS avg_num_prior_admissions
FROM
  readmissions_typed
GROUP BY
  admission_type
ORDER BY
  admission_type;

Running query in 'sqlite:///readmissions/readmissions.db'

admission_type,patients,readmitted_30days,readmission_30_day_rate,avg_time_in_hospital,avg_num_medications,avg_num_lab_procedures,avg_num_prior_admissions
elective,3037,821,27.0,5.0,24.87,49.74,1.04
emergency,5006,1366,27.0,4.99,25.11,49.84,1.0
urgent,1957,520,26.0,4.95,25.35,49.5,1.05


In [8]:
%%sql
SELECT
  insurance_type,
  COUNT(DISTINCT patient_id) AS patients,
  SUM(readmitted_within_30days) AS readmitted_30days,
  ROUND(
    100 * SUM(readmitted_within_30days) / COUNT(patient_id),
    2
  ) AS readmission_30_day_rate,
  ROUND(AVG(time_in_hospital), 2) AS avg_time_in_hospital,
  ROUND(AVG(num_medications), 2) AS avg_num_medications,
  ROUND(AVG(num_lab_procedures), 2) AS avg_num_lab_procedures,
  ROUND(AVG(num_prior_admissions), 2) AS avg_num_prior_admissions
FROM
  readmissions_typed
GROUP BY
  insurance_type
ORDER BY
  insurance_type;

Running query in 'sqlite:///readmissions/readmissions.db'

insurance_type,patients,readmitted_30days,readmission_30_day_rate,avg_time_in_hospital,avg_num_medications,avg_num_lab_procedures,avg_num_prior_admissions
medicaid,1468,431,29.0,5.01,24.84,49.44,1.05
medicare,3014,770,25.0,4.97,25.42,49.93,1.02
private,5058,1370,27.0,4.98,25.0,49.63,1.0
self-pay,460,136,29.0,5.0,24.6,50.64,1.16


In [17]:
%%sql
SELECT
  num_prior_admissions,
  COUNT(DISTINCT patient_id) AS patients,
  SUM(readmitted_within_30days) AS readmitted_30days,
  ROUND(
    100 * SUM(readmitted_within_30days) / COUNT(patient_id),
    2
  ) AS readmission_30_day_rate,
  ROUND(AVG(time_in_hospital), 2) AS avg_time_in_hospital,
  ROUND(AVG(num_medications), 2) AS avg_num_medications,
  ROUND(AVG(num_lab_procedures), 2) AS avg_num_lab_procedures
FROM
  readmissions_typed
GROUP BY
  num_prior_admissions
ORDER BY
  num_prior_admissions;

Running query in 'sqlite:///readmissions/readmissions.db'

num_prior_admissions,patients,readmitted_30days,readmission_30_day_rate,avg_time_in_hospital,avg_num_medications,avg_num_lab_procedures
0,4906,1314,26.0,4.98,24.78,50.07
1,2047,573,27.0,5.01,25.36,49.31
2,1521,426,28.0,4.99,25.13,50.11
3,1046,281,26.0,4.95,25.4,48.55
4,389,90,23.0,5.0,26.57,48.5
5,91,23,25.0,5.0,24.88,54.53


In [10]:
%%sql
SELECT
  r.primary_diagnosis_code,
  i.diagnosis_desc,
  COUNT(DISTINCT patient_id) AS patients,
  SUM(readmitted_within_30days) AS readmitted_30days,
  ROUND(
    100 * SUM(readmitted_within_30days) / COUNT(patient_id),
    2
  ) AS readmission_30_day_rate,
  ROUND(AVG(time_in_hospital), 2) AS avg_time_in_hospital,
  ROUND(AVG(num_medications), 2) AS avg_num_medications,
  ROUND(AVG(num_lab_procedures), 2) AS avg_num_lab_procedures,
  ROUND(AVG(num_prior_admissions), 2) AS avg_num_prior_admissions
FROM
  readmissions_typed r
  JOIN icd_10_chapter_codes i ON r.primary_diagnosis_code = i.chapter_code
GROUP BY
  diagnosis_desc
ORDER BY
  diagnosis_desc;

Running query in 'sqlite:///readmissions/readmissions.db'

primary_diagnosis_code,diagnosis_desc,patients,readmitted_30days,readmission_30_day_rate,avg_time_in_hospital,avg_num_medications,avg_num_lab_procedures,avg_num_prior_admissions
j45,Asthma,996,259,26.0,4.97,24.92,49.35,1.05
i25,Chronic ischemic heart disease,745,207,27.0,4.99,25.51,51.85,1.0
f32,Depressive episode,1033,281,27.0,4.97,25.59,49.52,0.99
e78,Disorders of lipoprotein metabolism and other lipidemias,1046,290,27.0,5.06,25.08,50.07,0.98
m54,Dorsalgia,990,267,26.0,4.93,25.44,48.69,1.05
i10,Essential (primary) hypertension,1157,294,25.0,5.01,25.19,50.03,1.02
k21,Gastro-esophageal reflux disease,1029,276,26.0,5.0,24.55,47.56,1.04
n39,Other disorders of urinary system,971,264,27.0,4.8,25.51,50.91,1.04
g47,Sleep disorders,821,230,28.0,4.97,24.99,49.9,1.07
e11,Type 2 diabetes mellitus,1212,339,27.0,5.1,24.33,50.04,1.01


Running query in 'sqlite:///readmissions/readmissions.db'

MAX(num_prior_admissions),MIN(num_prior_admissions)
5,0


Do we have any deceased patients in our dataset? These should be excluded from the analysis, since it is impossible for these patients to be readmitted.

In [19]:
%%sql
SELECT
  discharge_disposition,
  COUNT(*) AS disc_disp_count
FROM
  readmissions_typed
GROUP BY
  discharge_disposition
ORDER BY
  disc_disp_count DESC;

Running query in 'sqlite:///readmissions/readmissions.db'

discharge_disposition,disc_disp_count
home,7473
rehabilitation,1472
transfer,965
expired,90


Rather than altering the dataset, we can create a view of the data to conduct analysis without the sample of deceased patients.

In [5]:
%%sql
CREATE VIEW
  vw_living_readmissions AS
SELECT
  *
FROM
  readmissions_typed r
  JOIN icd_10_chapter_codes i ON r.primary_diagnosis_code = i.chapter_code
WHERE
  discharge_disposition != 'expired';

Running query in 'sqlite:///readmissions/readmissions.db'

++
||
++
++

Time to export our data for our next steps.

In [22]:
!sqlite3 -header -csv readmissions/readmissions.db "SELECT * FROM readmissions_typed r JOIN icd_10_chapter_codes i ON r.primary_diagnosis_code = i.chapter_code; " > readmissions_cleaned.csv